# Notebook 5 — SNR per file

**Purpose:** give every file in the Development Set a single number for how hard it is
acoustically, so the ablation results can be read against it (see `snr_vs_configs.ipynb`).

**Input:** the dataset itself (wav + sibling csv). **Nothing here needs a GPU or a run log.**

**Output:** `snr/per_file_snr.csv`, `snr/snr_by_subset.csv`, and two figures.

---

## What is being measured

There is no clean signal/noise reference in this dataset, so SNR is **estimated from the
annotations**: power inside annotated POS events is signal+noise, power in the
un-annotated stretches is the noise floor.

On STFT frames (`N_FFT=1024`, `HOP=256`, band-limited to 50–11025 Hz — the same front end
as `configs/train.yaml → features` and `data_preparation.py`):

| frame class | rule |
|---|---|
| **event** | frame centre falls inside a POS interval |
| **background** | frame is in *no* annotated interval, and outside a ±0.10 s guard band around every annotation |
| **excluded** | the guard bands themselves, and all UNK / NEG intervals |

The guard band keeps event onsets and offsets — which bleed across the annotation
boundary — out of the noise estimate.

With `P_ev` = mean event-frame power and `P_bg` = **median** background-frame power
(median, so a few passing cars or another species don't inflate the floor):

$$\mathrm{SNR_{dB}} = 10\log_{10}\frac{P_{ev} - P_{bg}}{P_{bg}}
\qquad
\mathrm{EBR_{dB}} = 10\log_{10}\frac{P_{ev}}{P_{bg}}$$

`SNR` subtracts the noise that is also present inside the event; `EBR` is the raw
event-to-background ratio, kept as a sanity column. Both are power *ratios*, so they are
invariant to the per-file peak normalisation (`y / max|y|`) that `data_preparation.py`
applies — this is the SNR the model sees.

Audio is streamed with `librosa.stream`: a full-file STFT of an hour-long recording at
`hop=256` would be a >1 GB matrix.


## 0 — Configuration

In [ ]:
import os
from pathlib import Path

# configs/train.yaml -> path.root_dir. Override without editing the notebook:
#   export DCASE_DATA_ROOT=/path/to/Dataset
DATA_ROOT = Path(os.environ.get("DCASE_DATA_ROOT", "/media/helios/Data/Uni/Masters/Dataset"))
DEV_SET = DATA_ROOT / "Development_Set"

# Test_Set is left out on purpose: only its first 5 events are annotated, so every
# later target call lands in the "background" pool and drags the noise floor up.
SPLITS = {
    "Training":   DEV_SET / "Training_Set",
    "Validation": DEV_SET / "Validation_Set",
}

# Front end -- identical to data_preparation.py / single_file_analysis.ipynb
SR_REF = 22050          # reference only; files are analysed at their native rate
N_FFT, HOP = 1024, 256
FMIN, FMAX = 50, 11025

GUARD_S = 0.10          # seconds masked either side of every annotation
MIN_BG_FRAMES = 100     # below this there is no usable noise estimate -> NaN
SNR_FLOOR_DB = -30.0    # clamp for events at or below the background median
BLOCK_FRAMES = 1024     # streaming block size, in STFT frames (~2 MB per block)
N_SHOTS = 5             # src/utils/evaluation.py:N_SHOTS -- the 5 shots the evaluator strips

OUT_DIR = Path("snr")
OUT_DIR.mkdir(exist_ok=True)

print(f"DATA_ROOT  {DATA_ROOT}   exists: {DATA_ROOT.is_dir()}")
for _name, _path in SPLITS.items():
    print(f"  {_name:<11}{_path}   exists: {_path.is_dir()}")

## 1 — Helpers

In [ ]:
import inspect
import traceback
import warnings

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

EPS = 1e-20
META_COLS = ["Audiofilename", "Starttime", "Endtime"]

# librosa.stream() only grew an `sr` argument in librosa 1.0, and in 1.1 its default
# becomes 22050 -- i.e. it starts resampling. Every frame index here is derived from
# sf.info().frames at the file's native rate, so pin sr=None where that is possible and
# rely on the native-rate behaviour of older versions where it is not.
try:
    _STREAM_SR = {"sr": None} if "sr" in inspect.signature(librosa.stream).parameters else {}
except (TypeError, ValueError):     # pragma: no cover -- unexpected librosa build
    _STREAM_SR = {}
print(f"librosa {librosa.__version__}"
      f"{' (stream sr pinned to native)' if _STREAM_SR else ' (stream reads native rate)'}")


def read_annotations(csv_path):
    """Return (pos, other) arrays of [start, end] in seconds.

    Same POS/UNK handling as single_file_analysis.ipynb, which works for both csv
    layouts in the dataset: Training_Set files carry one column per class, Validation
    files carry a single 'Q' column -- either way the label columns are everything
    that is not a META_COL.  (src/utils/evaluation.py:select_events_with_value is not
    used here: its find_positive_label() requires a 'Q' column and raises on the
    Training csvs.)

    Rows that are not POS -- UNK and NEG alike -- go to `other` and are excluded from
    both the event and the background pool.
    """
    df = pd.read_csv(csv_path, header=0, index_col=False)
    df.columns = [str(c).strip() for c in df.columns]
    class_cols = [c for c in df.columns if c not in META_COLS]
    labels = df[class_cols].astype(str).apply(lambda s: s.str.strip().str.upper())
    is_pos = (labels == "POS").any(axis=1).to_numpy()
    iv = df[["Starttime", "Endtime"]].astype(float).to_numpy()
    return iv[is_pos], iv[~is_pos]


def frame_times(n_frames, sr):
    """Centre time of every STFT frame, for center=False framing."""
    return (np.arange(n_frames) * HOP + N_FFT / 2) / sr


def frame_labels(t, pos, other):
    """0 = background, 1 = event, 2 = excluded (guard band, UNK/NEG)."""
    lab = np.zeros(len(t), np.int8)
    annotated = [iv for iv in (pos, other) if len(iv)]
    for s, e in (np.vstack(annotated) if annotated else np.empty((0, 2))):
        lab[(t >= s - GUARD_S) & (t <= e + GUARD_S)] = 2   # guard everything first,
    for s, e in pos:
        lab[(t >= s) & (t <= e)] = 1                       # then carve the events back out
    return lab


def event_mask(t, events):
    """Boolean frame mask covering the given [start, end] intervals."""
    m = np.zeros(len(t), bool)
    for s, e in events:
        m[np.searchsorted(t, s, "left"):np.searchsorted(t, e, "right")] = True
    return m


def scored_events(pos):
    """POS events the evaluator actually scores: those after the 5 shots.

    Mirrors src/utils/evaluation.py:remove_shots_from_ref, which drops every event
    ending at or before the end of the N_SHOTS-th positive event.
    """
    if len(pos) <= N_SHOTS:
        return np.empty((0, 2))
    cutoff = pos[N_SHOTS - 1][1]
    return pos[pos[:, 1] > cutoff]


def stream_power(wav_path):
    """Yield (bins, frames) power blocks, streamed at the file's native sample rate."""
    blocks = librosa.stream(
        str(wav_path),
        block_length=BLOCK_FRAMES,
        frame_length=N_FFT,
        hop_length=HOP,
        mono=True,
        dtype=np.float32,
        **_STREAM_SR,
    )
    for y in blocks:
        if len(y) < N_FFT:      # ragged tail, no complete frame left in it
            break
        # center=False is what makes streamed blocks line up frame-for-frame
        yield np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP, center=False)) ** 2


def to_db(num, den, floor=None):
    """10*log10(num/den) -- NaN for unusable inputs, optional dB floor."""
    if not np.isfinite(num) or not np.isfinite(den) or den <= 0:
        return np.nan
    value = 10 * np.log10(max(float(num), EPS) / float(den))
    return value if floor is None else max(value, floor)

In [ ]:
def analyse_file(wav_path, csv_path, keep_trace=False):
    """Estimate SNR for one file in a single streaming pass.

    Returns (metrics, spectra, trace): the per-file row, the mean event/background
    spectra (for the sanity-check figure), and optionally the frame-power trace.
    """
    info = sf.info(str(wav_path))
    sr = info.samplerate
    n_frames = 1 + max(0, info.frames - N_FFT) // HOP

    pos, other = read_annotations(csv_path)
    t = frame_times(n_frames, sr)
    lab = frame_labels(t, pos, other)

    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    band = (freqs >= FMIN) & (freqs <= min(FMAX, sr / 2))

    # ── one pass: per-frame in-band power, plus per-bin sums per frame class ───
    power = np.zeros(n_frames)
    sum_ev = np.zeros(int(band.sum()))
    sum_bg = np.zeros(int(band.sum()))
    n_ev = n_bg = 0
    i = 0
    for S in stream_power(wav_path):
        n = min(S.shape[1], n_frames - i)
        if n <= 0:
            break
        S = S[band, :n]
        power[i:i + n] = S.sum(axis=0)
        block = lab[i:i + n]
        ev, bg = block == 1, block == 0
        if ev.any():
            sum_ev += S[:, ev].sum(axis=1)
            n_ev += int(ev.sum())
        if bg.any():
            sum_bg += S[:, bg].sum(axis=1)
            n_bg += int(bg.sum())
        i += n
    power, lab, t = power[:i], lab[:i], t[:i]     # drop frames the stream never reached

    # ── the numbers ───────────────────────────────────────────────────────────
    p_ev = power[lab == 1].mean() if (lab == 1).any() else np.nan
    p_bg = np.median(power[lab == 0]) if n_bg >= MIN_BG_FRAMES else np.nan

    # Same statistic over the events the evaluator scores; the noise floor is still
    # the whole file's.  NaN when the file has 5 or fewer positive events.
    scored = scored_events(pos)
    m_scored = event_mask(t, scored) & (lab == 1)
    p_ev_scored = power[m_scored].mean() if m_scored.any() else np.nan

    metrics = {
        "filename": wav_path.name,
        "sr": sr,
        "duration_s": round(info.duration, 2),
        "n_events": len(pos),
        "n_events_scored": len(scored),
        "event_dur_median_s": float(np.median(pos[:, 1] - pos[:, 0])) if len(pos) else np.nan,
        "event_frac": float((lab == 1).mean()) if len(lab) else np.nan,
        "bg_frac": float((lab == 0).mean()) if len(lab) else np.nan,
        "n_bg_frames": n_bg,
        "p_ev_db": to_db(p_ev, 1.0),
        "p_bg_db": to_db(p_bg, 1.0),
        "snr_db": to_db(p_ev - p_bg, p_bg, floor=SNR_FLOOR_DB),
        "ebr_db": to_db(p_ev, p_bg),
        "snr_db_scored": to_db(p_ev_scored - p_bg, p_bg, floor=SNR_FLOOR_DB),
    }
    metrics["at_floor"] = bool(np.isfinite(metrics["snr_db"])
                               and metrics["snr_db"] <= SNR_FLOOR_DB)

    spectra = {
        "freqs": freqs[band],
        "event": sum_ev / n_ev if n_ev else np.full(int(band.sum()), np.nan),
        "background": sum_bg / n_bg if n_bg else np.full(int(band.sum()), np.nan),
    }
    trace = {"t": t, "power": power, "lab": lab, "pos": pos} if keep_trace else None
    return metrics, spectra, trace

## 2 — Walk the dataset

Same tree as `data_preparation.py:recursive_glob`, at unbounded depth. The **subset**
label is the folder holding the wav (`.../Validation_Set/PB24/BUK1_x.wav` → `PB24`),
which is how the evaluator derives subsets (see the `tools/build_subset_map.py`
docstring), so these labels line up with the per-subset results tables.

In [ ]:
def find_pairs(split_dir):
    """Split the split's wavs into (wav, csv) pairs and wavs with no csv."""
    pairs, orphans = [], []
    for wav in sorted(split_dir.rglob("*.wav")):
        csv = wav.with_suffix(".csv")
        if csv.exists():
            pairs.append((wav, csv))
        else:
            orphans.append(wav)
    return pairs, orphans


tasks, orphans = [], []
for split, split_dir in SPLITS.items():
    if not split_dir.is_dir():
        warnings.warn(f"{split} split not found at {split_dir} -- skipped")
        continue
    pairs, split_orphans = find_pairs(split_dir)
    tasks += [(split, wav.parent.name, wav, csv) for wav, csv in pairs]
    orphans += split_orphans

print(f"{len(tasks)} annotated files")
if orphans:
    print(f"{len(orphans)} wav(s) with no csv -- skipped:")
    for wav in orphans[:10]:
        print(f"   {wav}")

inventory = pd.DataFrame([(s, sub) for s, sub, _, _ in tasks], columns=["split", "subset"])
print()
print(inventory.value_counts(["split", "subset"]).sort_index().to_string())

## 3 — Compute

One streaming pass per file, so this is disk-bound. The whole Development Set takes a
few minutes on a warm disk.

In [ ]:
if not tasks:
    raise RuntimeError("No wav/csv pairs found -- check DATA_ROOT and the walk above.")

rows, failed = [], []

for split, subset, wav, csv in tqdm(tasks, desc="files"):
    try:
        metrics, _, _ = analyse_file(wav, csv)
    except Exception as exc:                     # keep going; every failure is reported
        failed.append((wav.name, f"{type(exc).__name__}: {exc}", traceback.format_exc()))
        continue
    metrics.update(split=split, subset=subset, path=str(wav))
    rows.append(metrics)

# Report failures BEFORE building the table: if every file failed, `rows` is empty and any
# column access below would raise a confusing KeyError instead of the actual cause.
if failed:
    print(f"{len(failed)} of {len(tasks)} file(s) failed:")
    for name, err, _ in failed[:10]:
        print(f"   {name}: {err}")
    if len(failed) > 10:
        print(f"   ... and {len(failed) - 10} more")
    print(f"\nFull traceback for the first failure ({failed[0][0]}):\n")
    print(failed[0][2])

if not rows:
    raise RuntimeError(f"All {len(tasks)} file(s) failed -- see the traceback above.")

COLS = ["split", "subset", "filename", "sr", "duration_s", "n_events", "n_events_scored",
        "event_dur_median_s", "event_frac", "bg_frac", "n_bg_frames",
        "p_ev_db", "p_bg_db", "snr_db", "ebr_db", "snr_db_scored", "at_floor", "path"]
df = pd.DataFrame(rows)[COLS].sort_values(["split", "subset", "filename"], ignore_index=True)
df.to_csv(OUT_DIR / "per_file_snr.csv", index=False)

print(f"{len(df)} files -> {OUT_DIR / 'per_file_snr.csv'}")
if df["at_floor"].any():
    print(f"\nclamped at the {SNR_FLOOR_DB:g} dB floor (events at or below the background median):")
    print("   " + ", ".join(df.loc[df["at_floor"], "filename"]))
if df["snr_db"].isna().any():
    print(f"\nno usable noise estimate (< {MIN_BG_FRAMES} background frames):")
    print("   " + ", ".join(df.loc[df["snr_db"].isna(), "filename"]))
df.head(10)

## 4 — Summary by subset

In [ ]:
q1 = lambda s: s.quantile(0.25)
q3 = lambda s: s.quantile(0.75)

summary = (df.groupby(["split", "subset"])
             .agg(files=("filename", "size"),
                  events=("n_events", "sum"),
                  hours=("duration_s", lambda s: s.sum() / 3600),
                  event_dur_s=("event_dur_median_s", "median"),
                  snr_med=("snr_db", "median"),
                  snr_q1=("snr_db", q1),
                  snr_q3=("snr_db", q3),
                  snr_min=("snr_db", "min"),
                  snr_max=("snr_db", "max"))
             .round(2)
             .sort_values("snr_med"))
summary.to_csv(OUT_DIR / "snr_by_subset.csv")

print(f"All {len(df)} files: median {df['snr_db'].median():.1f} dB   "
      f"IQR [{q1(df['snr_db']):.1f}, {q3(df['snr_db']):.1f}]   "
      f"range [{df['snr_db'].min():.1f}, {df['snr_db'].max():.1f}] dB\n")
summary

## 5 — Figures

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.4,
    "font.size": 10,
})

BLUE, ORANGE, INK, GREY = "#2a78d6", "#eb6834", "#3a3a3a", "#9a9a9a"
SPLIT_COLOR = {"Training": BLUE, "Validation": ORANGE}


def save(fig, name):
    fig.savefig(OUT_DIR / name, bbox_inches="tight", dpi=150)
    print(f"saved {OUT_DIR / name}")

In [ ]:
# ── Figure 1 -- per-file SNR by subset ────────────────────────────────────────
order = summary.reset_index().sort_values("snr_med")["subset"].tolist()
groups = [df.loc[df["subset"] == s, "snr_db"].dropna().to_numpy() for s in order]

fig, ax = plt.subplots(figsize=(1.05 * len(order) + 2.5, 4.2))
ax.boxplot(groups, positions=range(len(order)), widths=0.55, showfliers=False,
           medianprops=dict(color=INK, lw=1.6), boxprops=dict(color=GREY),
           whiskerprops=dict(color=GREY), capprops=dict(color=GREY))

rng = np.random.default_rng(1234)
for split, sub in df.groupby("split"):
    x = [order.index(s) + j for s, j in zip(sub["subset"], rng.uniform(-0.16, 0.16, len(sub)))]
    ax.scatter(x, sub["snr_db"], s=20, alpha=0.8, color=SPLIT_COLOR.get(split, INK),
               edgecolor="white", linewidth=0.5, zorder=3, label=split)

ax.axhline(0, color="#c8c8c8", lw=1, zorder=0)
for i, s in enumerate(order):
    ax.annotate(f"n={len(groups[i])}", (i, ax.get_ylim()[0]), textcoords="offset points",
                xytext=(0, 4), ha="center", fontsize=8, color="#7a7a7a")
ax.set_xticks(range(len(order)), order)
ax.set_xlabel("Subset (sorted by median SNR)")
ax.set_ylabel("Estimated SNR (dB)")
ax.set_title("Per-file event-to-background SNR", loc="left", fontsize=11)
if df["split"].nunique() > 1:
    ax.legend(frameon=False, fontsize=9, loc="upper left")
save(fig, "fig_snr_by_subset.png")
plt.show()

### Figure 2 — sanity check

Read this before trusting any number above. For the loudest and the quietest file:
the shaded POS events should sit visibly above the background median line, and the mean
event spectrum should separate from the background spectrum in the band where the species
calls. If they don't, the annotation parse or the framing is wrong.

In [ ]:
def sanity_check(filename, window_s=30):
    row = df[df["filename"] == filename].iloc[0]
    wav = Path(row["path"])
    _, spectra, trace = analyse_file(wav, wav.with_suffix(".csv"), keep_trace=True)
    t, power, lab, pos = trace["t"], trace["power"], trace["lab"], trace["pos"]

    t0 = max(0.0, pos[0][0] - 2) if len(pos) else 0.0
    win = (t >= t0) & (t <= t0 + window_s)
    db = 10 * np.log10(np.maximum(power, EPS))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
    ax1.plot(t[win], db[win], lw=0.7, color=INK)
    for s, e in pos:
        if e > t0 and s < t0 + window_s:
            ax1.axvspan(s, e, color=BLUE, alpha=0.18, lw=0)
    ax1.axhline(10 * np.log10(max(np.median(power[lab == 0]), EPS)),
                color=ORANGE, lw=1.4, label="background median")
    ax1.set_xlim(t0, t0 + window_s)
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("In-band power (dB, arb.)")
    ax1.set_title(f"{filename[:40]} -- POS events shaded", loc="left", fontsize=10)
    ax1.legend(frameon=False, fontsize=9)

    f = spectra["freqs"]
    ax2.plot(f, 10 * np.log10(np.maximum(spectra["event"], EPS)), lw=1.8, color=BLUE,
             label="event frames")
    ax2.plot(f, 10 * np.log10(np.maximum(spectra["background"], EPS)), lw=1.8, color=ORANGE,
             label="background frames")
    ax2.set_xlabel("Frequency (Hz)")
    ax2.set_ylabel("Mean power (dB, arb.)")
    ax2.set_title(f"{row['subset']} · SNR {row['snr_db']:.1f} dB · {row['n_events']} events",
                  loc="left", fontsize=10)
    ax2.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    return fig


usable = df.dropna(subset=["snr_db"])
for tag, filename in [("high", usable.loc[usable["snr_db"].idxmax(), "filename"]),
                      ("low",  usable.loc[usable["snr_db"].idxmin(), "filename"])]:
    save(sanity_check(filename), f"fig_snr_sanity_check_{tag}.png")
    plt.show()

## 6 — Build `file_to_subset.json` while the dataset is mounted

Not part of the SNR measurement, but this is the only notebook that runs where the
Validation reference tree exists, and `snr_vs_configs.ipynb` needs this map to be honest.
Without it `parse_logs.per_file_df` cannot backfill the files a log never printed (a log
only prints files that had at least one prediction — the seven runs printed 40, 41 and 42
of them), and it falls back to filename prefixes that merge `PB` with `PB24`. The files it
can't backfill are the F-measure = 0 files — exactly the ones the SNR analysis is about —
so leaving the map out biases the comparison upward and does it hardest in the low-SNR bin.

The map is `{filename.wav: {subset, n_pos_events}}`, built here from the columns already
computed above (`subset` is the folder name, `n_pos_events` is *all* POS events with no
5-shot stripping — the semantics the evaluator's backfill path uses).
`tools/build_subset_map.py` is the canonical builder and produces the same file; it is not
imported here because it pulls in `src.utils`, hence `pytorch_lightning`, which an
audio-only environment won't have.

> **This affects the other notebooks.** `results_comparison*.ipynb`,
> `per_file_analysis*.ipynb` and `threshold_analysis*.ipynb` all call `per_file_df`, so
> once this file exists their per-subset numbers change — they gain the missing total
> misses and split `PB`/`PB24`. That is the correction `tools/build_subset_map.py` exists
> to make, but it does mean any table already regenerated without it will move. Set
> `WRITE_SUBSET_MAP = False` to skip.

In [ ]:
import json
import warnings

WRITE_SUBSET_MAP = True
MAP_PATH = Path("file_to_subset.json")

if not WRITE_SUBSET_MAP:
    print("WRITE_SUBSET_MAP is False -- skipped")
elif MAP_PATH.exists():
    print(f"{MAP_PATH} already exists -- left alone "
          f"({len(json.loads(MAP_PATH.read_text()))} files)")
elif not SPLITS["Validation"].is_dir():
    print(f"Validation set not mounted -- cannot build {MAP_PATH}")
else:
    val = df[df["split"] == "Validation"].sort_values("path")

    # The logs, the evaluator's per-file results and this map are all keyed on the bare
    # filename, so files sharing a name cannot be told apart downstream. Keep the first
    # by path and say so, rather than stopping: the cost is that this one file's subset
    # and positive-event count come from one of the two copies.
    dupes = sorted(val.loc[val.duplicated("filename", keep=False), "filename"].unique())
    if dupes:
        warnings.warn(f"{len(dupes)} filename(s) repeat across subsets: {dupes}. "
                      "Keeping the first by path -- per-subset attribution for those is "
                      "ambiguous by construction.")
        for name in dupes:
            for i, r in enumerate(val[val["filename"] == name].itertuples()):
                print(f"   {'kept   ' if i == 0 else 'dropped'} {r.subset:<6} "
                      f"{r.n_events:>4} events  {r.path}")

    val = val.drop_duplicates("filename", keep="first")
    mapping = {r.filename: {"subset": r.subset, "n_pos_events": int(r.n_events)}
               for r in val.itertuples()}
    with open(MAP_PATH, "w") as fh:
        json.dump(mapping, fh, indent=1, sort_keys=True)
        fh.write("\n")

    counts = val["subset"].value_counts().sort_index()
    print(f"wrote {len(mapping)} files -> {MAP_PATH}  (commit it)")
    print("   " + ", ".join(f"{k}={v}" for k, v in counts.items()))
    print(f"   {int(val['n_events'].sum())} positive events in total")

## Caveats

- **`P_bg` is a noise floor, not silence.** The un-annotated stretches contain other
  species, rain, wind, machinery. Anything loud in there lowers the reported SNR — which
  is arguably the honest number for a detector, but it is not a clean-signal SNR.
- **Annotation-limited.** A file where the target species is present but unannotated
  reads as lower SNR, because those calls are pooled into the background.
- **dB values are relative.** `p_ev_db` / `p_bg_db` are in arbitrary units (STFT power,
  no window normalisation) and are comparable *within* a file, not across files. Only
  `snr_db` / `ebr_db` are meaningful across files.
- **Broadband.** Power is summed over 50–11025 Hz, so a narrowband call in a wide band
  looks quieter than it sounds. Two files with the same `snr_db` can differ a lot in how
  detectable the call is.
- **`snr_db` vs `snr_db_scored`.** `snr_db` uses every POS event; `snr_db_scored` uses
  only those after the 5 shots, which is the set the evaluator scores. They should rank
  files almost identically — if they don't for some file, that file's shots are not
  representative of its other events.

---

**Next:** `snr_vs_configs.ipynb` joins `snr/per_file_snr.csv` to the run logs.
